# 04 模型评估闭环：从回答到行为门禁

你现在已经知道 adapter 怎么训练、怎么用 `infer.py` 加载。

今天只解决一个问题：

> 我怎么判断一个 adapter 真的变好了，而不是 loss 好看、回答却变坏了？

这一本只看 4 个真实项目文件：

```text
scripts/compare_four_outputs.py
scripts/score_fixed_prompt_outputs.py
scripts/score_expanded_behavior_outputs.py
reports/stage5_structured_behavior_score_report.md
```

核心地图是：

```text
固定 prompt
  -> compare_four_outputs.py 生成多模型回答
  -> score_fixed_prompt_outputs.py 按透明规则打分
  -> reports/*.jsonl / *.csv / *.md 留证据
  -> 决定 adapter 接受、保留观察、还是拒绝
```

先把这条链路吃透，比先学一堆自动评测名词更有用。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/codex备份/qwen-local-lora-sft-dpo学习版")

print("项目根目录:", PROJECT_ROOT)

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=80):
    lines = read_text(rel).splitlines()
    print(f"--- {rel}:{start}-{min(end, len(lines))} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = [i for i, line in enumerate(lines, start=1) if keyword in line]
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits[:5]:
        show_file(rel, max(1, hit-context), min(len(lines), hit+context))
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 0. 今天只看哪些真实项目文件？

| 类型 | 路径 | 今天看它干嘛 |
|---|---|---|
| 固定题库 | `data/samples/custom_technical_prompts.jsonl` | 8 道核心行为考试题 |
| 多模型对比 | `scripts/compare_four_outputs.py` | 让 base / SFT / DPO 回答同一批题 |
| 固定打分 | `scripts/score_fixed_prompt_outputs.py` | 用规则判断回答是否踩中关键点 |
| 扩展打分 | `scripts/score_expanded_behavior_outputs.py` | 用 metadata 支持更多 held-out prompt |
| 评估报告 | `reports/stage5_structured_behavior_score_report.md` | 最终验收证据 |

先别急着上 LLM-as-judge。这个项目的 80% 评估理解，都在“固定 prompt + 可复现规则 + 人工解释”这条线上。

In [ ]:
for rel in [
    "data/samples/custom_technical_prompts.jsonl",
    "scripts/compare_four_outputs.py",
    "scripts/score_fixed_prompt_outputs.py",
    "scripts/score_expanded_behavior_outputs.py",
    "reports/stage5_structured_behavior_score_report.md",
]:
    print(f"{rel:60s}", "OK" if (PROJECT_ROOT / rel).exists() else "MISSING")

## 1. 先看考试题：固定 prompt 是行为门禁

入口：`data/samples/custom_technical_prompts.jsonl`

这里每一行都是一个固定问题。它不是为了聊天好玩，而是为了稳定暴露模型的核心坏习惯。

In [ ]:
print("固定 prompt 行数:", count_jsonl("data/samples/custom_technical_prompts.jsonl"))
for row in read_jsonl("data/samples/custom_technical_prompts.jsonl", n=3):
    print(json.dumps(row, ensure_ascii=False, indent=2)[:1000])
    print("---")

你要记住这句话：

> 固定 prompt 不是完整评测集，但它是项目的行为体检表。

如果一个新 adapter 连这些核心题都退步，就算训练 loss 很好看，也不能直接接受。

## 2. compare_four_outputs.py 做了什么？

它的作用很简单：同一批 prompt，让多个模型版本分别回答，然后写成 JSONL。

```text
prompt_file
  -> base_answer
  -> public_sft_answer
  -> custom_sft_v3_answer
  -> dpo_tiny_answer
  -> reports/compare_outputs_four_way_xxx.jsonl
```

In [ ]:
find_lines("scripts/compare_four_outputs.py", "parser.add_argument(\"--prompt_file\"", context=6)
find_lines("scripts/compare_four_outputs.py", "variants =", context=14)
find_lines("scripts/compare_four_outputs.py", "Saved four-way comparison", context=8)

这里最重要的不是“四方对比”这个数字，而是**同题对比**。

同一个 prompt 下，base、SFT、DPO 谁变好、谁退步，才看得清楚。

## 3. 安全打印评估命令：默认不跑模型

下面只打印命令，不执行推理。

如果你真的要跑，把 `RUN_COMPARE=True`。推理会占用显存，并且会写 report 文件。

In [ ]:
RUN_COMPARE = False

compare_cmd = [
    sys.executable, "scripts/compare_four_outputs.py",
    "--prompt_file", "data/samples/custom_technical_prompts.jsonl",
    "--public_adapter_path", "outputs/sft_lora_qwen05b_public",
    "--custom_adapter_path", "outputs/sft_lora_qwen05b_custom_v3_from_v1_patch",
    "--dpo_adapter_path", "outputs/dpo_lora_qwen05b_naive_v6",
    "--output_file", "reports/compare_outputs_four_way_my_eval.jsonl",
    "--max_new_tokens", "128",
    "--temperature", "0",
    "--local_files_only",
]

print(" ".join(compare_cmd))

if RUN_COMPARE:
    result = subprocess.run(compare_cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, encoding="utf-8", errors="replace")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("默认不运行。先理解：这一步会生成多模型回答 JSONL。")

## 4. score_fixed_prompt_outputs.py 做了什么？

它不是神秘评委，而是一套透明规则：

```text
每个 prompt 有一个 rubric
  -> required: 应该出现哪些关键概念
  -> forbidden: 不该出现哪些错误说法
  -> score = 命中 required - 惩罚 forbidden
  -> passed = 分数和禁忌项都过线
```

In [ ]:
find_lines("scripts/score_fixed_prompt_outputs.py", "class Criterion", context=8)
find_lines("scripts/score_fixed_prompt_outputs.py", "RUBRICS", context=8)
find_lines("scripts/score_fixed_prompt_outputs.py", "def score_answer", context=16)
find_lines("scripts/score_fixed_prompt_outputs.py", "def summarize", context=10)

你要抓住评估的本质：

- 不是看回答长不长。
- 不是看有没有中文。
- 不是看 loss 低不低。
- 是看它有没有说到项目关心的关键概念，并避开已经见过的坏说法。

## 5. 读取一份真实评估结果

先看已有的 Stage 5 汇总，不重新跑。

In [ ]:
show_file("reports/stage5_structured_behavior_score_report.md", start=1, end=80)

这个报告里最重要的结论是：

```text
custom-SFT v3: 7 / 8
DPO naive v6: 7 / 8
失败点仍然是 prompt 7: loss-vs-behavior
推荐 checkpoint 仍然是 outputs/sft_lora_qwen05b_custom_v3_from_v1_patch
```

这就是“训练指标”和“行为验收”分开的原因。

## 6. 如果要重新打分，命令是什么？

下面只打印命令。它会读取已有 compare JSONL，然后写新的 score JSONL / CSV / markdown。

In [ ]:
RUN_SCORE = False

score_cmd = [
    sys.executable, "scripts/score_fixed_prompt_outputs.py",
    "--input_files", "reports/compare_outputs_four_way_dpo_naive_v6.jsonl",
    "--output_jsonl", "reports/my_score_rows.jsonl",
    "--output_csv", "reports/my_score_rows.csv",
    "--report_file", "reports/my_score_report.md",
]

print(" ".join(score_cmd))

if RUN_SCORE:
    result = subprocess.run(score_cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, encoding="utf-8", errors="replace")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print("默认不运行。先理解：这一步把回答 JSONL 变成可复现的行为分数。")

## 7. 为什么还要 expanded behavior gate？

固定 8 题很有用，但也有风险：模型可能只学会某一道题的表述。

所以 Stage 5H 增加了扩展评估：保留原始 8 题，再加入 prompt 7 的改写和 held-out 题。

```text
固定 gate: 看核心能力有没有退步
expanded gate: 看模型是不是只记住了某个固定问法
```

In [ ]:
find_lines("scripts/score_expanded_behavior_outputs.py", "prompt_area", context=8)
find_lines("scripts/score_expanded_behavior_outputs.py", "Loss vs behavior", context=14)

for rel in [
    "data/samples/custom_technical_prompts_expanded_stage5h.jsonl",
    "reports/stage5i_expanded_behavior_score_v6_report.md",
]:
    print(f"{rel:65s}", "OK" if (PROJECT_ROOT / rel).exists() else "MISSING")

## 8. 评估闭环的接受标准

你可以把项目里的 adapter 分成三类：

| 类型 | 例子 | 怎么处理 |
|---|---|---|
| 推荐 checkpoint | `outputs/sft_lora_qwen05b_custom_v3_from_v1_patch` | 默认展示和面试叙事用它 |
| 最好 DPO 候选 | `outputs/dpo_lora_qwen05b_naive_v6` | 保留实验价值，但不替代推荐 checkpoint |
| 拒绝 artifact | v3 / Stage 5K / Stage 5O 等 | 记录失败原因，不当作最终成果 |

评估不是为了证明自己一定成功，而是为了知道什么时候该停。

## 9. 这一页你要能讲什么

> 本项目的模型评估不是只看 train loss 或 eval loss，而是先用 `compare_four_outputs.py` 在固定 prompt 上生成多模型回答，再用 `score_fixed_prompt_outputs.py` 按 required / forbidden 规则做透明行为打分，最后结合 markdown 报告决定 adapter 是否接受。DPO v6 的 preference 指标很好，但固定行为门禁仍然卡在 prompt 7，所以它是最好 DPO 候选，不是默认推荐 checkpoint。